## Bootstrap (click Run All — no setup required)

This cell makes the notebook self-installing. It:

1. Finds the Lunar-V2 repo root and puts it on `sys.path`.
2. Installs any missing third-party packages (`numpy`, `scipy`, `numba`, `matplotlib`, plus extras) into the current kernel.
3. Downloads any external data files this notebook needs.

You can re-run it any time; it's a no-op if everything is already present.

In [ ]:
# === Lunar-V2 notebook bootstrap — safe to re-run ===
import sys, pathlib
# Locate the repo root even if this notebook is opened from a weird CWD.
_here = pathlib.Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / 'pyproject.toml').is_file() and (_p / 'lunar' / '_bootstrap.py').is_file():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
else:
    raise RuntimeError('Could not find Lunar-V2 repo root from ' + str(_here))

from lunar import _bootstrap as boot
boot.ensure_lunar(extra=())


# Lunar-V2 Quickstart

This notebook walks through the core `lunar` library:

1. **Grid** — geometric depth grid construction
2. **Properties** — Hayne 2017 + Martinez-Siegler 2021 conductivity, specific heat, ice-coupled variants
3. **Solver** — 1D Crank-Nicolson pixel solver with Dirichlet and radiative BCs
4. **Illumination** — synthetic crater horizon trace
5. **Validation** — analytical thermal-wave check

For the full project goals, regolith-property citations, and project rules see `.claude/skills/SKILL.md` and `CLAUDE.md`.

**Author:** Ramon III Palinguba Gregorio (`rp3gregorio@gmail.com`)  
**Project:** TSUKIMI mission lunar thermal modeling pipeline (Kasai Lab, Institute of Science Tokyo)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from lunar import properties
from lunar.grid import make_geometric_grid
from lunar.solver import (
    PixelInputs,
    analytical_thermal_wave,
    solve_pixel,
)
from lunar.constants import LUNATION_SECONDS, Q_B_EQUATORIAL, SIGMA_SB

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 10})

## 1. Geometric depth grid

Uniform grids are forbidden (project rule #2). The `make_geometric_grid`
helper spaces layers with `dz[i+1] = dz[i] * (1 + growth)`, giving fine
resolution at the diurnal skin depth (~cm) and coarse resolution at the
thermal inertia depth (~m).

In [ ]:
grid = make_geometric_grid(z_max=3.0, dz0=0.002, growth=0.12)
print(f"n_layers = {grid.n_layers}")
print(f"top 5 dz [mm]: {np.round(grid.dz[:5] * 1e3, 3)}")
print(f"z_mid: surface={grid.z_mid[0]*1e3:.2f} mm, bottom={grid.z_mid[-1]:.3f} m")

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(grid.dz * 1e3, grid.z_mid, 'o-', ms=3)
ax.set(xlabel="layer thickness dz [mm]", ylabel="depth z [m]",
       xscale="log", title="Geometric depth grid")
ax.invert_yaxis()
ax.grid(alpha=0.3)
plt.show()

## 2. Regolith properties

Three conductivity models — Hayne (2017) H-parameter, Martinez &
Siegler (2021) density form, and the novel ice-coupled model — and two
specific-heat models (Hayne 2017 polynomial, Biele 2022 rational fit).

In [ ]:
T = np.linspace(40.0, 400.0, 200)
z_surface = np.zeros_like(T)

# Conductivity at z=0 for all three models (plus one with 30% pore ice).
K_hayne = properties.conductivity_hayne(T, z_surface)
K_mart = properties.conductivity_martinez(T, z=z_surface)
K_ice30 = properties.conductivity_icy(T, z_surface, phi_ice=np.full_like(T, 0.3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.semilogy(T, K_hayne, label="Hayne 2017 (H-param)")
ax1.semilogy(T, K_mart,  label="Martinez-Siegler 2021")
ax1.semilogy(T, K_ice30, label="Ice-coupled (phi=0.3)")
ax1.set(xlabel="T [K]", ylabel="K [W m$^{-1}$ K$^{-1}$]",
        title="Surface conductivity (z=0)")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

cp_hayne = properties.specific_heat(T, model="hayne")
cp_biele = properties.specific_heat(T, model="biele")
ax2.plot(T, cp_hayne, label="Hayne 2017 polynomial")
ax2.plot(T, cp_biele, label="Biele 2022 rational")
ax2.set(xlabel="T [K]", ylabel="$c_p$ [J kg$^{-1}$ K$^{-1}$]",
        title="Specific heat")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Analytical thermal-wave validation

For a semi-infinite domain with constant coefficients, a sinusoidal
surface temperature drives a decaying exponential wave,
$T(z,t) = T_0 + A\, e^{-z/\delta} \sin(\omega t - z/\delta)$ with
skin depth $\delta = \sqrt{2\alpha/\omega}$. This is the gold standard
for checking that the solver's Crank-Nicolson assembly, Thomas solve,
and Dirichlet BC are internally consistent.

In [ ]:
K_val, rho_val, cp_val = 1.0e-3, 1500.0, 600.0
alpha = K_val / (rho_val * cp_val)
T_mean, amplitude = 250.0, 80.0
P = LUNATION_SECONDS

wave_grid = make_geometric_grid(z_max=1.5, dz0=0.002, growth=0.08)
n_periods, n_t = 3, 721
t = np.linspace(0.0, n_periods * P, n_t)
omega = 2.0 * np.pi / P
T_forced = T_mean + amplitude * np.sin(omega * t)

def _const(val):
    def K(T_, z_): return np.full_like(z_, val, dtype=np.float64)
    return K

inputs = PixelInputs(
    grid=wave_grid, t=t, bc_mode="dirichlet",
    T_surface_forced=T_forced, Q_b=0.0,
    K_func=_const(K_val),
    rho_func=lambda z: np.full_like(z, rho_val, dtype=np.float64),
    cp_func=lambda T: np.full_like(T, cp_val, dtype=np.float64),
    T_init=np.full(wave_grid.n_layers, T_mean),
)
out = solve_pixel(inputs)

T_ana = analytical_thermal_wave(
    z=wave_grid.z_mid, t=t, T_mean=T_mean,
    amplitude=amplitude, period=P, alpha=alpha,
)
k0 = int(2 * (n_t - 1) / n_periods)
max_err = float(np.max(np.abs(out.T[:, k0:] - T_ana[:, k0:])))
rms_err = float(np.sqrt(np.mean((out.T[:, k0:] - T_ana[:, k0:])**2)))
print(f"max error = {max_err:.3f} K, rms error = {rms_err:.3f} K")
print(f"skin depth delta = {np.sqrt(2*alpha/omega)*100:.2f} cm")

sel = [0, 8, 16, 24]  # surface to a few skin depths
fig, ax = plt.subplots(figsize=(7, 4))
for k in sel:
    line, = ax.plot(t / P, out.T[k], lw=1.2, label=f"z = {wave_grid.z_mid[k]*100:.1f} cm")
    ax.plot(t / P, T_ana[k], '--', color=line.get_color(), lw=1, alpha=0.6)
ax.axvspan(2, 3, alpha=0.1, color='gray', label="validation window")
ax.set(xlabel="time [lunations]", ylabel="T [K]",
       title="Solver (solid) vs analytical thermal wave (dashed)")
ax.legend(fontsize=8, loc="upper right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Full radiative-BC pixel run

Sinusoidal insolation with the nonlinear radiative surface BC, spun up
over several lunations until `max |ΔT|` between cycles drops below the
convergence tolerance. Bottom BC is the equatorial geothermal flux.

In [ ]:
# Property callables for the full model (Martinez-Siegler K, Hayne rho, Hayne cp).
def K_full(T, z):
    return properties.conductivity_martinez(T, z=z)

def rho_full(z):
    return properties.density_hayne(z)

def cp_full(T):
    return properties.specific_heat(T, model="hayne")

grid_full = make_geometric_grid(z_max=3.0, dz0=0.002, growth=0.12)

# One lunation at 144 samples; spin up over 4 lunations for the demo
# (project rule: 10+ for a science run).
n_per = 144
n_t = 4 * n_per + 1
t_full = np.linspace(0.0, 4.0 * LUNATION_SECONDS, n_t)
# Equatorial-noon insolation: S = S0 * max(0, cos(phase))
phase = 2.0 * np.pi * (t_full / LUNATION_SECONDS)
insol = 1361.0 * np.maximum(0.0, np.cos(phase))

inputs_full = PixelInputs(
    grid=grid_full, t=t_full, bc_mode="radiative",
    insolation=insol, albedo=0.12, emissivity=0.95,
    Q_b=Q_B_EQUATORIAL,
    K_func=K_full, rho_func=rho_full, cp_func=cp_full,
    n_lunations_spinup=4, spinup_tol_K=0.05,
    T_init=np.full(grid_full.n_layers, 250.0),
)
out_full = solve_pixel(inputs_full)
print(f"spin-up cycles used: {out_full.n_spinup_cycles}, converged: {out_full.converged}")
print(f"surface T min/max: {out_full.T[0].min():.1f} / {out_full.T[0].max():.1f} K")
print(f"bottom T (z={grid_full.z_mid[-1]:.2f} m): {out_full.T[-1].mean():.2f} K")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Surface + a few depths over the last lunation
k_depths = [0, 6, 12, 20, 30, grid_full.n_layers - 1]
for k in k_depths:
    ax1.plot(t_full / LUNATION_SECONDS, out_full.T[k], lw=1.0,
             label=f"z = {grid_full.z_mid[k]*100:.1f} cm"
                   if grid_full.z_mid[k] < 1 else f"z = {grid_full.z_mid[k]:.2f} m")
ax1.set(xlabel="time [lunations]", ylabel="T [K]",
        title="Equatorial T(t) at selected depths")
ax1.legend(fontsize=7, loc="upper right"); ax1.grid(alpha=0.3)

# T(z) profiles at sampled phases in the final lunation
i_last = slice(3 * n_per, 4 * n_per + 1)
for frac, color in zip([0.0, 0.25, 0.5, 0.75], ["C0", "C1", "C2", "C3"]):
    idx = 3 * n_per + int(frac * n_per)
    ax2.plot(out_full.T[:, idx], grid_full.z_mid, color=color,
             label=f"phase = {frac:.2f}")
ax2.set(xlabel="T [K]", ylabel="depth z [m]", yscale="log",
        title="T(z) profiles across one lunation")
ax2.invert_yaxis()
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Synthetic-crater horizon tracer

The Mazarico (2011) horizon tracer marches rays outward from each pixel
and records the maximum elevation angle in each azimuth bin. On the
synthetic crater DEM, the center pixel should see its rim at the same
angle from every direction — a closed-form check for the tracer.

(This cell is skipped in the notebook by default because the Numba JIT
compile and ray-march take a few minutes on first run. Flip `RUN_HORIZON = True`
below to re-execute.)

In [ ]:
RUN_HORIZON = False

from lunar.illumination import (
    azimuth_bin_centers, compute_horizon, synthetic_crater_dem,
)

dem = synthetic_crater_dem(
    n=81, pixel_m=20.0, rim_radius_m=400.0,
    rim_height_m=200.0, rim_width_m=30.0,
)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(dem.elevation, cmap="terrain",
               extent=[dem.x[0], dem.x[-1], dem.y[-1], dem.y[0]])
fig.colorbar(im, ax=ax, label="elevation [m]")
ax.set(title="Synthetic crater DEM", xlabel="x [m]", ylabel="y [m]")
plt.show()

if RUN_HORIZON:
    horizon = compute_horizon(dem, n_azimuth=72, max_range_m=700.0, step_m=5.0)
    ci = dem.elevation.shape[0] // 2
    cj = dem.elevation.shape[1] // 2
    az = np.rad2deg(azimuth_bin_centers(72))
    expected = np.rad2deg(np.arctan2(200.0, 400.0))
    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(5, 5))
    ax.plot(np.deg2rad(az), np.rad2deg(horizon[ci, cj]), "o-", ms=3, label="tracer")
    ax.axhline(expected, color="r", ls="--", label=f"expected = {expected:.2f} deg")
    ax.set_title("Horizon profile at crater center")
    ax.legend(fontsize=8, loc="lower left", bbox_to_anchor=(1.05, 0))
    plt.show()
else:
    print("Horizon tracer skipped (RUN_HORIZON=False). Set True to recompute.")

## Next steps

This quickstart exercises the three pieces that are already implemented:
1D CN solver, regolith-property models, and the horizon tracer. The
remaining pipeline stages (full DEM ingestion, polar illumination with
SPICE ephemeris, view factors, TSUKIMI RTM coupling, ice-stability
maps) are scaffolded but not yet wired. See the project summary in
`README.md` and the agent files under `.claude/skills/agents/`.